# Week 2 exercise (Option B) — Education assistant with OpenAI Agents SDK + OpenRouter



In [ ]:
!pip install python-decouple

## Imports, OpenRouter model, agent without tools

In [ ]:
import os
from typing import Any

import httpx
import wikipedia
import gradio as gr
from decouple import config
from dotenv import load_dotenv
from openai import AsyncOpenAI

from agents import Agent, Runner, OpenAIChatCompletionsModel, function_tool

load_dotenv()

OPENROUTER_BASE = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL = "openai/gpt-4o-mini"
OPENROUTER_API_KEY = config("OPENROUTER_API_KEY")
NEWS_API_KEY = config("NEWS_API_KEY")


or_client = AsyncOpenAI(
    base_url=OPENROUTER_BASE,
    api_key=OPENROUTER_API_KEY,
)

or_model = OpenAIChatCompletionsModel(model=OPENROUTER_MODEL, openai_client=or_client)
print("Model:", OPENROUTER_MODEL)

In [ ]:
basic_agent = Agent(
    name="TutorBasic",
    instructions="You are a concise educational tutor. Answer in plain language. No tools yet.",
    model=or_model,
    tools=[],
)

smoke = await Runner.run(basic_agent, "Say hello in one short sentence.")
print(smoke.final_output)

## Wikipedia tool

In [ ]:
wikipedia.set_lang("en")


@function_tool
def wikipedia_summary(topic: str, sentences: int = 4) -> dict[str, Any]:
    """Encyclopedic overview for stable facts, history, science, biographies. Not for breaking news."""
    topic = (topic or "").strip()
    if not topic:
        return {"error": "empty topic"}
    try:
        page = wikipedia.page(topic, auto_suggest=True)
        summary = wikipedia.summary(topic, sentences=min(max(sentences, 1), 8), auto_suggest=True)
        return {"title": page.title, "url": page.url, "summary": summary}
    except wikipedia.DisambiguationError as e:
        return {"error": "disambiguation", "options": e.options[:8]}
    except Exception as e:
        return {"error": type(e).__name__, "message": str(e)}

In [ ]:
tutor_wiki_only = Agent(
    name="TutorWiki",
    instructions=(
        "You are an educational assistant. For background topics (definitions, history, science concepts), "
        "call wikipedia_summary first, then explain using that text and cite title + URL. "
        "If the tool errors, say so honestly."
    ),
    model=or_model,
    tools=[wikipedia_summary],
)

wiki_test = await Runner.run(tutor_wiki_only, "What is photosynthesis? One paragraph.")
print(wiki_test.final_output[:800] if wiki_test.final_output else wiki_test)

## News + DuckDuckGo tools, full tutor agent

In [ ]:
@function_tool
async def news_search(query: str, page_size: int = 5) -> dict[str, Any]:
    """Recent news when the user asks about current events, today, this week, or breaking stories."""
    api_key = NEWS_API_KEY
    if not api_key:
        return {
            "error": "NEWS_API_KEY not set",
            "hint": "Add NEWS_API_KEY from newsapi.org to .env to enable this tool.",
        }
    query = (query or "").strip()
    if not query:
        return {"error": "empty query"}
    page_size = max(1, min(int(page_size), 10))
    try:
        async with httpx.AsyncClient(timeout=30.0) as client:
            r = await client.get(
                "https://newsapi.org/v2/everything",
                params={
                    "q": query,
                    "language": "en",
                    "sortBy": "publishedAt",
                    "pageSize": page_size,
                    "apiKey": api_key,
                },
            )
        data = r.json()
        if data.get("status") != "ok":
            return {"error": data.get("message", "newsapi error")}
        articles = []
        for a in data.get("articles", []):
            articles.append(
                {
                    "title": a.get("title"),
                    "url": a.get("url"),
                    "source": (a.get("source") or {}).get("name"),
                    "publishedAt": a.get("publishedAt"),
                    "description": a.get("description"),
                }
            )
        return {"query": query, "articles": articles}
    except Exception as e:
        return {"error": type(e).__name__, "message": str(e)}


@function_tool
async def duckduckgo_instant_answer(query: str) -> dict[str, Any]:
    """Quick instant-answer snippet when Wikipedia is unclear or you need a short fact. No API key."""
    query = (query or "").strip()
    if not query:
        return {"error": "empty query"}
    try:
        async with httpx.AsyncClient(timeout=20.0) as client:
            r = await client.get(
                "https://api.duckduckgo.com/",
                params={"q": query, "format": "json", "no_html": 1},
            )
        d = r.json()
        out = {
            "query": query,
            "abstract": d.get("Abstract") or "",
            "abstract_url": d.get("AbstractURL") or "",
            "heading": d.get("Heading") or "",
        }
        snippets = []
        for t in (d.get("RelatedTopics") or [])[:5]:
            if isinstance(t, dict) and "Text" in t:
                snippets.append({"text": t.get("Text"), "url": t.get("FirstURL")})
        out["related"] = snippets
        if not out["abstract"] and not snippets:
            out["note"] = "No instant answer; try wikipedia_summary."
        return out
    except Exception as e:
        return {"error": type(e).__name__, "message": str(e)}

In [ ]:
tutor_agent = Agent(
    name="EducationTutor",
    instructions=(
        "You are a helpful educational assistant.\n"
        "- Use news_search for timely/current events (today, this week, breaking news).\n"
        "- Use wikipedia_summary for stable concepts, history, science, biographies.\n"
        "- Use duckduckgo_instant_answer for quick facts or when other tools are weak.\n"
        "Ground answers in tool output; cite titles and URLs. If a tool returns an error dict, explain it honestly."
    ),
    model=or_model,
    tools=[wikipedia_summary, news_search, duckduckgo_instant_answer],
)

## Gradio chat 

`Runner.run` accepts a **list of message dicts** (see e.g. `agent_manager_refactor/deep_research.py` in community contributions).

In [ ]:
async def chat(message: str, history: list) -> str:
    """Gradio ChatInterface type='messages': history is list of {role, content}."""
    msgs = list(history) + [{"role": "user", "content": message}]
    result = await Runner.run(tutor_agent, msgs)
    return result.final_output or ""


gr.ChatInterface(
    chat,
    type="messages",
    title="Education tutor (Agents SDK + OpenRouter)",
).launch()

### Sample prompts (paste into the chat)

1. **Wikipedia:** "Explain what mitochondria do for a high-school student."
2. **News (needs `NEWS_API_KEY`):** "What are recent headlines about climate conferences?"
3. **Ambiguous / quick fact:** "What is the capital of Australia?" (may use Wikipedia or DDG)

In [ ]:
# Optional: run one turn from code (no Gradio)
demo_msgs = [{"role": "user", "content": "In two sentences, what is the water cycle?"}]
demo = await Runner.run(tutor_agent, demo_msgs)
print(demo.final_output)